<a href="https://colab.research.google.com/github/LINWOO0099/ml-leakage-pipeline--senthamil-/blob/main/Clean_the_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Dataset
data = {
    'customerID':     ['C001','C002','C003','C004','C005','C006','C007','C008',
                       'C009','C010','C011','C012','C013','C014','C015','C016',
                       'C017','C018','C019','C020'],
    'gender':         ['Male','Female','Male','Female','Male','Female','Male','Female',
                       'Male','Female','Male','Female','Male','Female','Male','Female',
                       'Male','Female','Male','Female'],
    'SeniorCitizen':  [0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1],
    'tenure':         [12,34,2,45,1,22,8,60,15,3,27,50,7,18,42,5,30,55,10,24],
    'ContractType':   ['Month-to-month','One year','Month-to-month','Two year',
                       'Month-to-month','One year','Month-to-month','Two year',
                       'Month-to-month','Month-to-month','One year','Two year',
                       'Month-to-month','Month-to-month','Two year','Month-to-month',
                       'One year','Two year','Month-to-month','One year'],
    'MonthlyCharges': [70.5,55.0,85.3,42.0,95.1,60.0,78.4,38.5,88.0,72.0,
                       52.0,35.0,91.2,65.0,40.0,80.0,58.0,33.0,76.5,62.0],
    'TotalCharges':   ['846.0','1870.0',' ','1890.0','95.1','1320.0','627.2','2310.0',
                       '1320.0','216.0','1404.0','1750.0','638.4','1170.0','1680.0',
                       '400.0','1740.0','1815.0','765.0','1488.0'],
    'Churn':          ['Yes','No','Yes','No','Yes','No','Yes','No','Yes','Yes',
                       'No','No','Yes','Yes','No','Yes','No','No','Yes','No']
}

df = pd.DataFrame(data)

# ---------------------------
# Task 1 — Clean the Data
# ---------------------------

# Drop customerID
df.drop(columns=['customerID'], inplace=True)

# Replace blank spaces with NaN
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Fill missing values with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


# ---------------------------
# Task 2 — Encode, Split, Scale
# ---------------------------

# One-hot encoding
df = pd.get_dummies(df, columns=['gender', 'ContractType'], drop_first=True)

# Features & target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ---------------------------
# Task 3 — Train and Evaluate
# ---------------------------

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Accuracy
train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
test_acc = accuracy_score(y_test, model.predict(X_test_scaled))

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)


# ---------------------------
# Task 4 — Threshold Tuning
# ---------------------------

# Probabilities
probs = model.predict_proba(X_test_scaled)[:, 1]

# Threshold 0.5
pred_05 = (probs >= 0.5).astype(int)
count_05 = pred_05.sum()

# Threshold 0.3
pred_03 = (probs >= 0.3).astype(int)
count_03 = pred_03.sum()

print("\nCustomers flagged as churners:")
print("Threshold 0.5:", count_05)
print("Threshold 0.3:", count_03)

